# Task: PINN for ODE #9

**Ordinary differential equation (ODE #9):**
- $(x+1)(y\,y' - 1) = y^2$,  $x \in [0, 2]$
- Initial condition: $y(0) = 1$
- Exact solution: $y = e^x - \dfrac{1}{x+2}$

We build and train a PINN to approximate the solution, visualize the learning curve, evaluate MAPE/MSPE, and compare with the exact solution.

## 1. Imports and configuration

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from math import *
import tensorflow as tf

# Add directory containing pinns.py to path (lab8 folder)
_cwd = os.getcwd()
for _path in [_cwd, os.path.join(_cwd, 'lab8')]:
    if _path not in sys.path:
        sys.path.insert(0, _path)

tf.keras.backend.set_floatx('float64')
tf.get_logger().setLevel('ERROR')

from pinns import PINNs

## 2. Domain and data

- Domain: $x \in [0, 2]$  
- Initial condition: $y(0) = 1$  
- Collocation points for solving the ODE

In [ ]:
# Domain x in [0, 2]
x_min, x_max = 0.0, 2.0
nx = 200
x_domain = np.linspace(x_min, x_max, nx)
X_star = x_domain.reshape(-1, 1)

# Initial condition: y(0) = 1
X_init = np.array([[0.0]])
u_init = np.array([[1.0]])

# Collocation points (inside the domain)
N_colloc = 500
np.random.seed(42)
idx_colloc = np.random.choice(X_star.shape[0], min(N_colloc, X_star.shape[0]), replace=False)
X_colloc_train = X_star[idx_colloc]

print("Number of collocation points:", X_colloc_train.shape[0])
print("Initial condition: x_init =", X_init[0, 0], ", y(0) =", u_init[0, 0])

## 3. PINN definition for the ODE

Residual of the equation $(x+1)(y\,y' - 1) = y^2$:  
$\mathcal{R}(x) = (x+1)(\hat{y}\,\hat{y}' - 1) - \hat{y}^2$

In [ ]:
def net_transform(X_f, model_nn):
    """Network output = approximation y(x)."""
    return model_nn(X_f)

def f_user(X_f, model_nn):
    """ODE residual: (x+1)(y*y' - 1) - y^2 = 0."""
    x_temp = X_f[:, 0:1]
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x_temp)
        u = model_nn(x_temp)
    u_x = tape.gradient(u, x_temp)
    del tape
    # (x+1)*(y*y' - 1) - y^2
    residual = (x_temp + 1.0) * (u * u_x - 1.0) - u**2
    return residual

def loss_f(f):
    return tf.reduce_mean(tf.square(f))

## 4. Network architecture and training

In [ ]:
# Input: x (1 variable), output: y (1 variable)
layers = [1] + [64] * 4 + [1]
lr = 0.001

model_classic = PINNs(
    X_colloc_train,
    net_transform,
    f_user,
    loss_f,
    layers,
    lr,
    type_problem='forward',
    X_init=X_init,
    u_init=u_init
)

model_classic.train(max_epochs_adam=5000, max_epochs_lbfgs=3000, print_per_epochs=500)

## 5. Learning curve (loss evolution)

In [ ]:
plt.figure(figsize=(8, 4))
plt.semilogy(model_classic.loss_array, color='steelblue', linewidth=1.5)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (log scale)', fontsize=12)
plt.title('PINN learning curve (loss evolution)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Prediction and exact solution

In [ ]:
def y_exact(x):
    """Exact solution: y = e^x - 1/(x+2)."""
    x = np.asarray(x)
    return np.exp(x) - 1.0 / (x + 2.0)

# PINN prediction over the full domain
X_eval = tf.convert_to_tensor(X_star, dtype='float64')
y_pinn = model_classic.net_transform(X_eval, model_classic.net_u)
y_pinn_np = y_pinn.numpy().flatten()

# Exact solution
y_exact_np = y_exact(X_star.flatten())

## 7. MAPE and MSPE metrics

In [ ]:
# MAPE: Mean Absolute Percentage Error (%)
# MAPE = (1/n) * sum( |y_true - y_pred| / |y_true| ) * 100
# Avoid division by zero using max(|y_true|, eps)
eps = 1e-10
abs_y_true = np.maximum(np.abs(y_exact_np), eps)
mape = np.mean(np.abs(y_exact_np - y_pinn_np) / abs_y_true) * 100

# MSPE: Mean Squared Percentage Error (%)
# MSPE = (1/n) * sum( ((y_true - y_pred) / y_true)^2 ) * 100
mspe = np.mean(((y_exact_np - y_pinn_np) / abs_y_true) ** 2) * 100

print("Deviation from exact solution:")
print("  MAPE = {:.6f} %".format(mape))
print("  MSPE = {:.6f} %".format(mspe))

## 8. Visualization: exact solution vs PINN approximation

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(X_star, y_exact_np, 'b-', label='Exact solution $y = e^x - 1/(x+2)$', linewidth=2)
plt.plot(X_star, y_pinn_np, 'r--', label='PINN approximation', linewidth=1.5)
plt.scatter(X_init, u_init, color='green', s=80, zorder=5, label='Initial condition $y(0)=1$')
plt.xlabel('$x$', fontsize=12)
plt.ylabel('$y$', fontsize=12)
plt.title('ODE #9: exact solution and PINN approximation')
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()